In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 26. LoRA and Pruning — Parameter-Efficient Fine-Tuning [CPU/GPU Benchmark ⑫]

> **Learning Goals**
> - Derive the low-rank approximation mathematics of LoRA
> - Understand pruning and sparsity
> - Measure how much faster LoRA training is than full fine-tuning

## 26.1 Why PEFT Is Needed

Full fine-tuning updates every parameter, so it requires large GPU memory and is slow.

**PEFT** (Parameter-Efficient Fine-Tuning) updates only a small subset:
- LoRA, Adapter, Prefix Tuning, Prompt Tuning
- Often reaches performance close to full fine-tuning with less than 1% of parameters

## 26.2 LoRA — Low-Rank Adaptation

Assume the weight update $\Delta W$ is low-rank:
$$W' = W + \Delta W = W + B A$$

- $A \in \mathbb{R}^{r \times d}$, $B \in \mathbb{R}^{d \times r}$
- $r \ll d$ (usually 4-64)
- Parameters: $d^2 \to 2rd$, a large saving

Initialization:
- $A \sim \mathcal{N}(0, \sigma^2)$ (random)
- $B = 0$ so initially $\Delta W = 0$

Scaling: $\Delta W = \frac{\alpha}{r} B A$

Training: freeze $W$ and train only $A,B$.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class LoRALinear(nn.Module):
    """Linear layer with LoRA applied."""
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # base Weight frozen
        self.base.weight.requires_grad = False
        if bias:
            self.base.bias.requires_grad = False
        # LoRA adapter
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        # base + LoRA delta
        base_out = self.base(x)
        lora_out = (x @ self.A.t()) @ self.B.t() * self.scaling
        return base_out + lora_out

# Parameter Count Comparison
d = 4096
full = nn.Linear(d, d)
lora = LoRALinear(d, d, r=8, alpha=16)
full_params = sum(p.numel() for p in full.parameters())
lora_params = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Full Linear: {full_params:,} params")
print(f"LoRA (r=8): {lora_params:,} trainable params")
print(f"Ratio: {lora_params/full_params*100:.3f}%")
print("\n=> LoRA trains only 0.4% of parameters!")


## 26.3 Effect of Rank $r$

Larger $r$ increases expressiveness but also increases memory. Choose $r$ according to task complexity.
- Simple tasks: $r = 4$
- More complex fine-tuning: $r = 16, 32, 64$


In [ ]:
# parameter count and performance by rank (simulation)
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
d = 4096
print(f"{'r':>5} | {'LoRA params':>12} | {'% of full':>10}")
print("-" * 35)
for r in ranks:
    lora_params = 2 * r * d  # A + B
    full_params = d * d
    print(f"{r:>5} | {lora_params:>12,} | {lora_params/full_params*100:>9.3f}%")

# Visualization
fig, ax = plt.subplots(figsize=(9, 5))
params_pct = [2 * r * d / (d * d) * 100 for r in ranks]
ax.plot(ranks, params_pct, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('Trainable params (%)')
ax.set_title(f'LoRA: rank vs trainable params (d={d})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch26_lora_ranks.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.4 Where Should LoRA Be Applied?

Typical locations in a transformer:
- $W_q, W_k, W_v, W_o$ in attention
- $W_1, W_2$ in the FFN

Common practice applies LoRA to attention Q and V; QLoRA-style setups may apply it more broadly. The placement affects performance.


In [ ]:
# attentionin LoRA apply
class LoRAAttention(nn.Module):
    def __init__(self, d_model, n_heads, r=8):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # base weights (frozen)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_qkv.weight.requires_grad = False
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.W_o.weight.requires_grad = False
        # LoRA adapter (QKV only)
        self.lora_qkv_A = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.lora_qkv_B = nn.Parameter(torch.zeros(3 * d_model, r))
        self.scaling = 16 / r

    def forward(self, x, mask=None):
        B, T, D = x.shape
        base = self.W_qkv(x)
        # LoRA delta
        delta = (x @ self.lora_qkv_A.t()) @ self.lora_qkv_B.t() * self.scaling
        qkv = base + delta
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# test
d, h = 64, 4
attn = LoRAAttention(d, h, r=4)
x = torch.randn(2, 10, d)
out, w = attn(x)
print(f"Output: {out.shape}")
n_trainable = sum(p.numel() for p in attn.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in attn.parameters())
print(f"Trainable: {n_trainable:,} / Total: {n_total:,} ({n_trainable/n_total*100:.2f}%)")


## 26.5 Pruning

### Unstructured Pruning
Set individual weights to zero:
$$\tilde{W}_{ij} = W_{ij} \cdot \mathbb{1}[|W_{ij}| > \tau]$$

Problem: the model is sparse, but hardware acceleration is difficult.

### Structured Pruning
Remove channels or heads:
- Remove low-importance attention heads
- Remove filters/channels in CNNs
- More hardware-friendly

**Lottery Ticket Hypothesis**: dense networks contain sparse subnetworks that can achieve strong performance.


In [ ]:
# pruning example
def magnitude_prune(W, sparsity=0.5):
    """magnitude-based unstructured Pruning."""
    threshold = torch.quantile(W.abs().flatten(), sparsity)
    mask = (W.abs() > threshold).float()
    return W * mask, mask

# Test
torch.manual_seed(0)
W = torch.randn(100, 100) * 0.1
for sparsity in [0.5, 0.7, 0.9, 0.95]:
    W_pruned, mask = magnitude_prune(W, sparsity)
    actual_sparsity = 1 - mask.mean()
    print(f"target sparsity {sparsity}: real {actual_sparsity:.4f}, "
          f"remaining elements {int(mask.sum())}")

# visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
W_orig = torch.randn(50, 50) * 0.1
for ax, sp in zip(axes, [0.0, 0.7, 0.9]):
    W_p, _ = magnitude_prune(W_orig, sp)
    ax.imshow(W_p.numpy(), cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.set_title(f'Sparsity = {sp}')
plt.tight_layout()
plt.savefig('../figures/ch26_pruning.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.6 [CPU/GPU Benchmark ⑫] Full FT vs LoRA vs QLoRA


In [ ]:
# Full FT vs LoRA comparison
from llm_math.bench import time_fn

# Synthetic model with large linear layers
class TinyModel(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

# Full FT
def make_full_ft():
    model = TinyModel()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

# LoRA
class TinyModelLoRA(nn.Module):
    def __init__(self, d=512, r=8):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
        # base frozen
        for p in self.parameters():
            p.requires_grad = False
        # LoRA adapter: do not replace base fc1/fc2/fc3 with LoRA versions
        # add only the delta on top of the base)
        self.lora_A1 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B1 = nn.Parameter(torch.zeros(d * 4, r))
        self.lora_A2 = nn.Parameter(torch.randn(r, d * 4) * 0.01)
        self.lora_B2 = nn.Parameter(torch.zeros(d, r))
        self.lora_A3 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B3 = nn.Parameter(torch.zeros(d, r))
        self.scaling = 16 / r
    def forward(self, x):
        # fc1 + LoRA delta
        h1 = F.relu(self.fc1(x) + (x @ self.lora_A1.t()) @ self.lora_B1.t() * self.scaling)
        # fc2 + LoRA delta
        h2 = F.relu(self.fc2(h1) + (h1 @ self.lora_A2.t()) @ self.lora_B2.t() * self.scaling)
        # fc3 + LoRA delta
        out = self.fc3(h2) + (h2 @ self.lora_A3.t()) @ self.lora_B3.t() * self.scaling
        return out

def make_lora():
    model = TinyModelLoRA(r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

# one-step training time comparison
def bench_step(model, opt, x, y, loss_fn, device='cpu'):
    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

d = 256
loss_fn = nn.MSELoss()
x = torch.randn(8, d)
y = torch.randn(8, d)

# Generate a model with d=256
def make_full_ft(d=256):
    model = TinyModel(d=d)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

def make_lora(d=256):
    model = TinyModelLoRA(d=d, r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

model_full, opt_full = make_full_ft(d=d)
model_lora, opt_lora = make_lora(d=d)

n_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Full FT: {n_full:,} trainable params")
print(f"LoRA:    {n_lora:,} trainable params ({n_lora/n_full*100:.2f}%)")

res_full = time_fn(bench_step(model_full, opt_full, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
res_lora = time_fn(bench_step(model_lora, opt_lora, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
print(f"\nFull FT 1 step: {res_full['mean_ms']:.3f} ms")
print(f"LoRA 1 step:    {res_lora['mean_ms']:.3f} ms")
print(f"Speed Improvement: {res_full['mean_ms'] / res_lora['mean_ms']:.2f}x")
print("\n=> LoRA LoRA has fewer trainable parameters, so both memory and speed improve.")


## 26.7 Key Takeaways

| Method | Trainable Parameters | Memory | Performance |
|---|---|---|---|
| Full FT | 100% | high | best |
| LoRA | ~1% | low | similar |
| QLoRA | ~1% + 4-bit base | very low | slight loss |
| Adapter | ~5% | moderate | similar |
| Prefix Tuning | ~0.1% | very low | slight loss |

## Exercises

1. Train with LoRA ranks $r = 1, 4, 16, 64$ and compare performance.
2. Compare applying LoRA to all QKV projections versus Q only.
3. Compare accuracy after pruning 50%, 70%, and 90% of weights.
4. Measure training time and memory for Full FT vs LoRA.
5. Explain why the low-rank assumption behind LoRA is meaningful.

> Solutions: `solutions/ch26_solutions.ipynb`
